# Tokenization

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

/Users/asadbek/Documents/LLM/tokenization/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer.tokenize("Hello how are you")
# Ġ - bo'sh joy ni anglatadi

['Hello', 'Ġhow', 'Ġare', 'Ġyou']

In [3]:
tokenizer.tokenize("Hellohow are you")


['Hell', 'oh', 'ow', 'Ġare', 'Ġyou']

In [4]:
tokenizer.tokenize("Hello how are you")


['Hello', 'Ġhow', 'Ġare', 'Ġyou']

In [5]:
tokenizer.encode("Hello")

[15496]

In [6]:
tokenizer.encode("Hello world")


[15496, 995]

In [7]:
# Bu xato. encode ga text berilishi kerak.
text = "Hello world"
tokens = tokenizer.tokenize(text)

ids = tokenizer.encode(tokens)
print(tokens)
print(ids)


['Hello', 'Ġworld']
[[15496], [128, 254, 6894]]


In [8]:
# 1. text to ids
text = "Hello world"

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)   # tokens emas, text beriladi

print(tokens)
print(ids)

['Hello', 'Ġworld']
[15496, 995]


In [9]:
# 2. tokens to ids
text = "Hello world"

tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)   # tokens uchun maxsus funksiya

print(tokens)
print(ids)

['Hello', 'Ġworld']
[15496, 995]


In [10]:
# 3 tokens to string
print(tokenizer.convert_tokens_to_string(tokens))

Hello world


In [13]:
from transformers import AutoModel

gpt2_model = AutoModel.from_pretrained("gpt2")
embeddinggemma_2 = AutoModel.from_pretrained("google/embeddinggemma-2")

Loading weights: 100%|██████████| 1376/1376 [00:00<00:00, 10114.44it/s]


# GPT2

In [14]:
gpt2_model

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
)

In [16]:
gpt2_model.wte

Embedding(50257, 768)

In [17]:
gpt2_model.wpe

Embedding(1024, 768)

In [18]:
gpt2_model.drop

Dropout(p=0.1, inplace=False)

In [22]:
gpt2_model.h[0].attn.c_attn

Conv1D(nf=2304, nx=768)

In [ ]:
wte_embedding = gpt2_model.wte
wte_embedding
# Token → vektor
# 50 257 (lug’at hajmi)
# Input token_ids 

Embedding(50257, 768)

In [ ]:
wpe_embedding = gpt2_model.wpe
wpe_embedding
# Pozitsiya → vektor
# 1024 (maksimal uzunlik)
# Pozitsiya raqamlari (0, 1, 2, ...)

Embedding(1024, 768)

In [32]:
# List
text = "Assalomu alaykum"
tokens = tokenizer.tokenize(text)
print(tokens)
print(tokenizer.encode(text))

['Ass', 'al', 'om', 'u', 'Ġal', 'ay', 'k', 'um']
[8021, 282, 296, 84, 435, 323, 74, 388]


In [ ]:
# Tensor
inputs = tokenizer(text, return_tensors="pt") # tokenize, ids, pytorch - pt tensor
print(inputs)
print(inputs["input_ids"])
print(inputs["input_ids"].shape) # [1, 8] => 1->matn soni, 8->token ids
print(inputs["attention_mask"])

{'input_ids': tensor([[8021,  282,  296,   84,  435,  323,   74,  388]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
tensor([[8021,  282,  296,   84,  435,  323,   74,  388]])
torch.Size([1, 8])
tensor([[1, 1, 1, 1, 1, 1, 1, 1]])


In [50]:
wte_embeddings = wte_embedding(inputs["input_ids"])
print("wte")
print(wte_embeddings.shape) # [1, 8, 768] => 1 - matnlar soni, 8 - token ids, 768 - har bir tokenni ifodalovchi sonlar hajmi
print(wte_embeddings[0][3][:10])

wte
torch.Size([1, 8, 768])
tensor([ 0.0539,  0.0709,  0.1180, -0.0946,  0.0358,  0.0356, -0.2607,  0.0131,
         0.0252,  0.1109], grad_fn=<SliceBackward0>)


In [67]:
import torch

ids = inputs["input_ids"]  
n = ids.shape[1]  

positions = torch.arange(n).unsqueeze(0) # [[0, 1, 2, ..., n-1]]
wpe_embeddings = wpe_embedding(positions)
print("wpe")
print(positions)
print(wpe_embeddings.shape)
print(wpe_embeddings[0][3][:10])


wpe
tensor([[0, 1, 2, 3, 4, 5, 6, 7]])
torch.Size([1, 8, 768])
tensor([-0.0003, -0.0738,  0.1055,  0.0007, -0.0156,  0.0107,  0.0114,  0.0599,
        -0.0132,  0.0086], grad_fn=<SliceBackward0>)


# EmbeddingGemma 2

In [74]:
tokenizer_EG = AutoTokenizer.from_pretrained("google/embeddinggemma-2")

In [ ]:
embeddinggemma_2

EmbeddingGemma2Model(
  (vision_tower): Gemma4VisionModel(
    (patch_embedder): Gemma4VisionPatchEmbedder(
      (input_proj): Linear(in_features=768, out_features=768, bias=False)
    )
    (encoder): Gemma4VisionEncoder(
      (rotary_emb): Gemma4VisionRotaryEmbedding()
      (layers): ModuleList(
        (0-15): 16 x Gemma4VisionEncoderLayer(
          (self_attn): Gemma4VisionAttention(
            (q_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=768, out_features=768, bias=False)
            )
            (k_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=768, out_features=768, bias=False)
            )
            (v_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=768, out_features=768, bias=False)
            )
            (o_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=768, out_features=768, bias=False)
            )
            (q_norm): Gemma4RMSNorm()
            (k_norm

In [85]:
embeddinggemma_2.language_model

EmbeddingGemma2TextModel(
  (embed_tokens): EmbeddingGemma2TextScaledWordEmbedding(262144, 512, padding_idx=0)
  (layers): ModuleList(
    (0-4): 5 x EmbeddingGemma2EncoderLayer(
      (self_attn): EmbeddingGemma2Attention(
        (q_proj): Linear(in_features=512, out_features=1024, bias=False)
        (k_proj): Linear(in_features=512, out_features=512, bias=False)
        (v_proj): Linear(in_features=512, out_features=512, bias=False)
        (o_proj): Linear(in_features=1024, out_features=512, bias=False)
        (q_norm): EmbeddingGemma2RMSNorm()
        (k_norm): EmbeddingGemma2RMSNorm()
        (v_norm): EmbeddingGemma2RMSNorm()
      )
      (mlp): EmbeddingGemma2MLP(
        (gate_proj): Linear(in_features=512, out_features=2048, bias=False)
        (up_proj): Linear(in_features=512, out_features=2048, bias=False)
        (down_proj): Linear(in_features=2048, out_features=512, bias=False)
        (act_fn): GELUTanh()
      )
      (input_layernorm): EmbeddingGemma2RMSNorm()
   

In [84]:
embedding = embeddinggemma_2.language_model.embed_tokens
embedding

EmbeddingGemma2TextScaledWordEmbedding(262144, 512, padding_idx=0)

In [ ]:
text = "I like to sleep"
tokenizer_EG.tokenize(text)
# _ => bosh joyni anglatadi

['I', '▁like', '▁to', '▁sleep']

In [79]:
inputs = tokenizer_EG(text, return_tensors="pt")
tensor_ids = inputs["input_ids"]
tensor_ids

tensor([[     2, 236777,   1133,    531,   6745,      1]])

In [89]:
embedding(tensor_ids).shape

torch.Size([1, 6, 512])